First, import all necessary packages

In [2]:
import anndata as ad
import pandas as pd
import numpy as np
import MINGLE as mg

/opt/anaconda3/lib/python3.13/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/opt/anaconda3/lib/python3.13/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/opt/anaconda3/lib/python3.13/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


Read in data

In [4]:
file_path = r"/Volumes/data/MINGLE/Data/Melanoma/melanoma_all_information.csv"
cells = mg.pp.read_file(file_path)

/opt/anaconda3/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [5]:
cluster_col = "Cell_Type"
neighborhood_col = "Neighborhood"
region_key = "region"

Next, we perform our centroid calculation with mg.tl.centroid_Calculation. By default, cluster_col is set to "cell_type" and neighborhood_col has a default parameter of "neighborhood". However, we can pass our own arguments into the function.

In [6]:
centroids = mg.tl.centroid_Calculation(cells, cluster_col=cluster_col, neighborhood_col=neighborhood_col, region_col=region_key)

Example dummy cols: ['Cell_Type__B', 'Cell_Type__Bcatenin+ Tumor', 'Cell_Type__CD163+ CD206+ Macrophage', 'Cell_Type__CD4+ T', 'Cell_Type__CD4+ Treg']


Then, let's look at our results from centroids!

In [7]:
centroids.X

array([[5.61423838e-01, 1.40036297e+00, 2.95409888e-01, ...,
        1.71805825e-02, 1.33297092e-03, 3.93608585e-02],
       [9.60060954e-02, 3.54521602e-01, 1.81107163e+00, ...,
        1.90561805e-02, 6.38032041e-04, 2.52453070e-02],
       [7.35625178e-02, 3.11155826e-01, 2.46928871e-01, ...,
        2.08292864e-02, 3.21584783e-04, 1.79284010e-02],
       ...,
       [1.31544694e-01, 4.42223012e-01, 5.89473426e-01, ...,
        5.00270072e-03, 8.50936049e-04, 3.08220480e-02],
       [1.19595379e-01, 4.11614180e-01, 3.16023320e-01, ...,
        2.55227052e-02, 5.46328432e-04, 2.34641470e-02],
       [7.98668712e-02, 3.18161458e-01, 2.86973774e-01, ...,
        2.49087326e-02, 1.41841336e-03, 3.95475887e-02]])

Next, we run our Gaussian Mixture Model to calculate probabilities for cells across all buckets. The cpu-based version is used in this tutorial.

In [ ]:
mg.tl.cpu_gmm_probability(CELLS_ADATA=cells, CENTROIDS_ADATA=centroids, cluster_col=cluster_col, neighborhood_col=neighborhood_col, region_key=region_key)

Example dummy cols: ['Cell_Type__B', 'Cell_Type__Bcatenin+ Tumor', 'Cell_Type__CD163+ CD206+ Macrophage', 'Cell_Type__CD4+ T', 'Cell_Type__CD4+ Treg']
Using 14 processes.


/opt/anaconda3/lib/python3.13/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/opt/anaconda3/lib/python3.13/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/opt/anaconda3/lib/python3.13/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/opt/anaconda3/lib/python3.13/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/opt/anaconda3/lib/python3.13/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprec

Now we run mg.tl.findPositives() to count the number of probabilities that exceed a predetermined threshold. This function has a few notable parameters: prob_key -- which is how you access the probabilities in AnnData.obsm, threshold -- which is the probability cutoff, and result_key -- which is where the counts are placed in AnnData.obs.

By default, prob_key = "neighborhood_probabilities", threshold = 0.25, result_key = "Count_Above_Threshold"

In [ ]:
mg.tl.findPositives(cells)